# Notebook 6 — Portfolio Backtest

Builds equal-weight top-five strategies from model probabilities, compares them with an equal-weight 25-portfolio benchmark, and reports annualized return, volatility, Sharpe ratio, and maximum drawdown.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

In [2]:
MODEL_ROOT=Path("../models")
FEATURE_ROOT=Path("../data/features")
RESULT_ROOT=Path("../results")

In [3]:
RUN_EXPERIMENTS = [
    "monthly_ff3",
    "monthly_ff5",
    "daily_ff3",
    "daily_ff5",
]

EXPERIMENT_CONFIG = {
    "monthly_ff3": {"frequency": "monthly", "factor_model": "ff3", "window": 12, "periods_per_year": 12},
    "monthly_ff5": {"frequency": "monthly", "factor_model": "ff5", "window": 12, "periods_per_year": 12},
    "daily_ff3": {"frequency": "daily", "factor_model": "ff3", "window": 60, "periods_per_year": 252},
    "daily_ff5": {"frequency": "daily", "factor_model": "ff5", "window": 60, "periods_per_year": 252},
}

unknown = set(RUN_EXPERIMENTS) - set(EXPERIMENT_CONFIG)
if unknown:
    raise ValueError(f"Unknown experiments: {sorted(unknown)}")

In [4]:
PROBABILITY_COLUMNS={
    "XGBoost Signature":"xgb_signature_prob","XGBoost GARCH":"xgb_garch_prob","XGBoost Combined":"xgb_combined_prob",
    "CNN Signature":"cnn_signature_prob","CNN GARCH":"cnn_garch_prob","CNN Combined":"cnn_combined_prob",
    "Ensemble Signature":"ensemble_signature_prob","Ensemble GARCH":"ensemble_garch_prob","Ensemble Combined":"ensemble_combined_prob",
}

def build_top_k_strategy(data, probability_column, k=5):
    rows=[]
    for date,g in data.groupby("Date"):
        selected=g.nlargest(k,probability_column)
        rows.append({"Date":date,"StrategyReturn":selected["RealizedNextReturn"].mean(),
                     "SelectedPortfolios":", ".join(selected["Portfolio"].astype(str))})
    return pd.DataFrame(rows).sort_values("Date").reset_index(drop=True)

def performance_statistics(returns, periods_per_year):
    r=pd.to_numeric(pd.Series(returns),errors="coerce").dropna()
    if (r<=-1).any(): raise ValueError("Return at or below -100%; check return units.")
    wealth=(1+r).cumprod(); total=wealth.iloc[-1]-1
    annual=wealth.iloc[-1]**(periods_per_year/len(r))-1
    vol=r.std(ddof=1)*np.sqrt(periods_per_year)
    sharpe=r.mean()/r.std(ddof=1)*np.sqrt(periods_per_year) if r.std(ddof=1)>0 else np.nan
    dd=wealth/wealth.cummax()-1
    return {"Total Return":total,"Annualized Return":annual,"Annualized Volatility":vol,"Sharpe Ratio":sharpe,"Maximum Drawdown":dd.min()}

In [5]:
for experiment in RUN_EXPERIMENTS:
    cfg=EXPERIMENT_CONFIG[experiment]
    model_dir=MODEL_ROOT/experiment; feature_dir=FEATURE_ROOT/experiment; out_dir=RESULT_ROOT/experiment/"backtest"
    out_dir.mkdir(parents=True,exist_ok=True)
    if not (model_dir/"test_probabilities.csv").exists() or not (feature_dir/"meta_test.csv").exists():
        print("Skipping incomplete experiment:",experiment); continue
    predictions=pd.read_csv(model_dir/"test_probabilities.csv")
    meta=pd.read_csv(feature_dir/"meta_test.csv",parse_dates=["Date","RealizationDate"])
    data=pd.concat([meta.reset_index(drop=True),predictions.reset_index(drop=True)],axis=1)
    data["RealizedNextReturn"]=pd.to_numeric(data["RealizedNextReturn"],errors="coerce")

    # Kenneth French files usually store returns in percent. Convert only when magnitude indicates percent units.
    if data["RealizedNextReturn"].abs().quantile(0.95)>1:
        data["RealizedNextReturn"]=data["RealizedNextReturn"]/100.0
    data=data.dropna(subset=["RealizedNextReturn"])
    if (data["RealizedNextReturn"]<=-1).any():
        raise ValueError(f"{experiment}: invalid return at or below -100% after conversion")

    strategies={name:build_top_k_strategy(data,col,5) for name,col in PROBABILITY_COLUMNS.items()}
    records=[]
    plt.figure(figsize=(13,7))
    for name,strategy in strategies.items():
        strategy["CumulativeValue"]=(1+strategy["StrategyReturn"]).cumprod()
        stats=performance_statistics(strategy["StrategyReturn"],cfg["periods_per_year"]); stats["Model"]=name; records.append(stats)
        plt.plot(strategy["Date"],strategy["CumulativeValue"],label=name)
        strategy.to_csv(out_dir/(name.lower().replace(" ","_")+"_strategy.csv"),index=False)

    benchmark=data.groupby("Date",as_index=False)["RealizedNextReturn"].mean().rename(columns={"RealizedNextReturn":"StrategyReturn"})
    benchmark["CumulativeValue"]=(1+benchmark["StrategyReturn"]).cumprod()
    stats=performance_statistics(benchmark["StrategyReturn"],cfg["periods_per_year"]); stats["Model"]="Equal-Weight 25 Portfolios"; records.append(stats)
    plt.plot(benchmark["Date"],benchmark["CumulativeValue"],"--",linewidth=2,label="Equal-Weight 25 Portfolios")
    plt.title(f"Growth of $1 — {experiment}"); plt.legend(fontsize=8); plt.tight_layout(); plt.savefig(out_dir/"cumulative_wealth.png",dpi=200); plt.close()

    perf=pd.DataFrame(records)[["Model","Total Return","Annualized Return","Annualized Volatility","Sharpe Ratio","Maximum Drawdown"]].sort_values("Sharpe Ratio",ascending=False)
    perf.to_csv(out_dir/"performance_summary.csv",index=False)
    benchmark.to_csv(out_dir/"equal_weight_benchmark.csv",index=False)
    print("Backtested",experiment,"best:",perf.iloc[0]["Model"])

Backtested monthly_ff3 best: CNN GARCH
Backtested monthly_ff5 best: Ensemble Signature
Backtested daily_ff3 best: Equal-Weight 25 Portfolios
Backtested daily_ff5 best: Equal-Weight 25 Portfolios
